# Ingestão da YouTube Data API

Notebook de exploração do módulo `youtube_etl_genai.youtube_client`. Ele coleta um canal, seus vídeos recentes, comentários e respostas, e exibe os resultados como DataFrames Spark.

Para a ingestão de produção, execute o Job da wheel: ele lê `control.video_targets`, atualiza os registros atuais e grava `video_snapshots` e `channel_snapshots`. Este notebook não substitui esse fluxo operacional.

Antes de executar: instale o wheel do projeto como biblioteca do cluster/job e configure um segredo Databricks com a chave da API.

O wheel do projeto deve estar declarado como biblioteca do cluster ou Job. Não instale dependências manualmente neste notebook.

In [1]:
# Job parameters remain strings so this notebook mirrors Databricks task inputs.
dbutils.widgets.text("channel_handle", "@LuizSayãoOficial", "Canal")
dbutils.widgets.text("max_videos", "5", "Quantidade de vídeos")
dbutils.widgets.text("secret_scope", "youtube_api_key", "Secret scope")
dbutils.widgets.text("secret_key", "api-key", "Secret key")

Box(children=(Label(value='Canal'), Text(value='@LuizSayãoOficial')))

Box(children=(Label(value='Quantidade de vídeos'), Text(value='5')))

Box(children=(Label(value='Secret scope'), Text(value='youtube_api_key')))

Box(children=(Label(value='Secret key'), Text(value='api-key')))

In [2]:
import json
from datetime import datetime, timezone
from typing import Any
from uuid import uuid4

channel_handle = dbutils.widgets.get("channel_handle")
max_videos = int(dbutils.widgets.get("max_videos"))
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")
api_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)

if max_videos < 1:
    raise ValueError("max_videos deve ser maior que zero")

ingestion_id = str(uuid4())
run_started_at = datetime.now(timezone.utc)
run_schema = (
    "ingestion_id string, channel_handle string, channel_name string, started_at timestamp, "
    "ended_at timestamp, status string, error_message string"
)

In [3]:
(spark.createDataFrame(
    [(ingestion_id, channel_handle, channel_handle, run_started_at, None, "RUNNING", None)],
    run_schema,
).write.mode("append").format("delta").saveAsTable(
    "youtube_lakehouse.control.ingestion_runs"
))

In [5]:
import sys
from pathlib import Path

# In a Databricks Job the wheel supplies this package. For local notebook
# runs, add the repository's src/ directory when the project is not installed.
_path = Path.cwd()
for _parent in (_path, *_path.parents):
    _src = _parent / "src"
    if (_src / "youtube_etl_genai").is_dir():
        sys.path.insert(0, str(_src))
        break
else:
    raise ModuleNotFoundError(
        "Não foi possível localizar src/youtube_etl_genai; instale o projeto"
        " ou execute o notebook a partir do repositório."
    )

from youtube_etl_genai.youtube_client import YouTubeAPIError, YouTubeClient

api_responses: list[dict[str, Any]] = []


def capture_response(
    resource: str, params: dict[str, Any], response: dict[str, Any]
) -> None:
    """Keep the original API payload for audit and reprocessing."""
    api_responses.append({
        "ingestion_id": ingestion_id,
        "resource": resource,
        "request_params_json": json.dumps(params, sort_keys=True),
        "response_json": json.dumps(response, sort_keys=True),
        "received_at": datetime.now(timezone.utc),
    })


client = YouTubeClient(
    api_key=api_key,
    timeout=30,
    response_observer=capture_response,
)

In [6]:
channel_raw = client.get_channel_by_handle(channel_handle)
if not channel_raw:
    raise RuntimeError(f"Canal não encontrado: {channel_handle}")

channel = client.normalize_channel(channel_raw)
channel

{'channel_id': 'UCtkOT5oDa4pasg0haswhCCw',
 'title': 'Luiz Sayão',
 'description': 'Sou teólogo, linguista e hebraísta (mestre pela USP), atuando como conferencista internacional e professor na área bíblica no Brasil e no exterior. Trabalhei elaboração e coordenação da tradução de versões da Bíblia como a NVI 2000, Almeida 21 e A Mensagem. Escrevi diversos livros e criei o comentário bíblico Rota 66 e a Bíblia de Estudo Rota 66. Além disso, fui diretor do Seminário Batista do Sul do Brasil, pastor da Igreja Batista Nações Unidas (SP) e idealizador da Bíblia Brasileira de Estudo e da Bíblia de Estudo Esperança. Atualmente, sou conselheiro acadêmico na Faculdade Batista Pioneira e Mentor Ministerial do Seminário Batista do Sul do Brasil.\n',
 'custom_url': '@luizsayaooficial',
 'published_at': '2020-05-27T20:54:06.452026Z',
 'country': 'BR',
 'view_count': 39729028,
 'subscriber_count': 283000,
 'video_count': 1345,
 'uploads_playlist_id': 'UUtkOT5oDa4pasg0haswhCCw'}

In [7]:
playlist_items = []
for position, item_raw in enumerate(client.iter_uploads(channel["uploads_playlist_id"])):
    if position >= max_videos:
        break
    playlist_items.append(client.normalize_playlist_item(item_raw))

video_ids = [item["video_id"] for item in playlist_items if item.get("video_id")]
videos_raw = client.get_videos(video_ids)
videos = [client.normalize_video(video) for video in videos_raw]
print(f"Vídeos coletados: {len(videos)}")
videos

Vídeos coletados: 5


[{'video_id': 'GgkkpNgPNl0',
  'channel_id': 'UCtkOT5oDa4pasg0haswhCCw',
  'channel_title': 'Luiz Sayão',
  'title': 'O significado da advertência de Hebreus | Luiz Sayão',
  'description': 'Seja membro deste canal e ganhe benefícios:\nhttps://www.youtube.com/channel/UCtkOT5oDa4pasg0haswhCCw/join\n\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - \n\nMeu novo livro 365 TORRENTES NO DESERTO já está disponível no site!\n\nhttps://luizsayao.com/produto/2776/?v=3cb56c81f4b8\n\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - \n\nREDES SOCIAIS\n\nInstagram:\n@luizsayao \n\nFacebook:\n@luizsayao\n\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - \n\nCURSOS\n\n\nO Mistério das Origens:\nhttps://luizsayao.com/cursos/os-misterios-das-origens/\n\nHebraico Bíblico:\nhttps://luizsayao.com/cursos/hebraico-biblico-iniciante-2025/\n\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -\n\nE-mail para contato:\nsayaocontato@gmail.com',
  'publi

In [8]:
comments: list[dict[str, Any]] = []
replies: list[dict[str, Any]] = []

for video in videos:
    video_id = video["video_id"]
    try:
        # The API rejects comment requests when comments are disabled.
        for thread_raw in client.iter_comment_threads(video_id):
            comment = client.normalize_top_level_comment(thread_raw)
            comments.append(comment)
            if comment["reply_count"] > 0:
                for reply_raw in client.iter_replies(comment["comment_id"]):
                    replies.append(client.normalize_reply(reply_raw))
    except YouTubeAPIError as exc:
        print(f"Comentários indisponíveis para {video_id}: {exc}")

print(f"Comentários: {len(comments)} | Respostas: {len(replies)}")

Comentários: 34 | Respostas: 18


In [9]:
# Explicit schemas prevent Spark Connect from inferring nullable fields
# such as parent_id from empty or partially populated collections.
from pyspark.sql.types import (
    ArrayType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

api_responses_schema = StructType([
    StructField("ingestion_id", StringType(), nullable=False),
    StructField("resource", StringType(), nullable=False),
    StructField("request_params_json", StringType(), nullable=False),
    StructField("response_json", StringType(), nullable=False),
    StructField("received_at", TimestampType(), nullable=False),
])

channels_schema = StructType([
    StructField("channel_id", StringType(), nullable=True),
    StructField("title", StringType(), nullable=True),
    StructField("description", StringType(), nullable=True),
    StructField("custom_url", StringType(), nullable=True),
    StructField("published_at", StringType(), nullable=True),
    StructField("country", StringType(), nullable=True),
    StructField("view_count", LongType(), nullable=True),
    StructField("subscriber_count", LongType(), nullable=True),
    StructField("video_count", LongType(), nullable=True),
    StructField("uploads_playlist_id", StringType(), nullable=True),
])

videos_schema = StructType([
    StructField("video_id", StringType(), nullable=True),
    StructField("channel_id", StringType(), nullable=True),
    StructField("channel_title", StringType(), nullable=True),
    StructField("title", StringType(), nullable=True),
    StructField("description", StringType(), nullable=True),
    StructField("published_at", StringType(), nullable=True),
    StructField("category_id", StringType(), nullable=True),
    StructField("tags", ArrayType(StringType()), nullable=True),
    StructField("duration", StringType(), nullable=True),
    StructField("definition", StringType(), nullable=True),
    StructField("caption", StringType(), nullable=True),
    StructField("view_count", LongType(), nullable=True),
    StructField("like_count", LongType(), nullable=True),
    StructField("comment_count", LongType(), nullable=True),
    StructField("privacy_status", StringType(), nullable=True),
])

comments_schema = StructType([
    StructField("thread_id", StringType(), nullable=True),
    StructField("comment_id", StringType(), nullable=True),
    StructField("parent_id", StringType(), nullable=True),
    StructField("video_id", StringType(), nullable=True),
    StructField("author_name", StringType(), nullable=True),
    StructField("author_channel_id", StringType(), nullable=True),
    StructField("text", StringType(), nullable=True),
    StructField("like_count", LongType(), nullable=True),
    StructField("published_at", StringType(), nullable=True),
    StructField("updated_at", StringType(), nullable=True),
    StructField("reply_count", LongType(), nullable=True),
])

replies_schema = StructType([
    StructField("comment_id", StringType(), nullable=True),
    StructField("parent_id", StringType(), nullable=True),
    StructField("video_id", StringType(), nullable=True),
    StructField("author_name", StringType(), nullable=True),
    StructField("author_channel_id", StringType(), nullable=True),
    StructField("text", StringType(), nullable=True),
    StructField("like_count", LongType(), nullable=True),
    StructField("published_at", StringType(), nullable=True),
    StructField("updated_at", StringType(), nullable=True),
])

In [ ]:
# from pyspark.sql import functions as F

channel_df = spark.createDataFrame([channel], schema=channels_schema)

display(channel_df)

In [11]:
videos_df = spark.createDataFrame(videos, schema=videos_schema)

display(videos_df.select("video_id", "title", "published_at", "view_count"))

,video_id,title,published_at,view_count
0,GgkkpNgPNl0,O significado da advertência de Hebreus | Luiz Sayão,2026-08-02T23:00:39Z,368
1,ZzrZAX4UQf4,Pra quê ter filhos e família se o mundo é tão mal | Luiz Sayão,2026-08-02T00:11:47Z,5392
2,u00KKIKgatE,Tragédias e Terremotos sinalizam o fim do mundo agora? | Luiz Sayão e Podcrê,2026-08-01T23:00:19Z,3839
3,mzYnTZpD_GI,Judeu que segue Jesus deixa de ser Judeu? | Luiz Sayão,2026-08-01T00:00:22Z,5017
4,EUAEaqDfexo,Pra quê o Templo de Salomão? | Luiz Sayão,2026-07-31T23:00:13Z,3864


In [ ]:
comments_df = spark.createDataFrame(comments, schema=comments_schema)

display(comments_df)

In [ ]:
replies_df = spark.createDataFrame(replies, schema=replies_schema)

display(replies_df)

## Persistência em Delta

Esta célula persiste os dados normalizados em `silver`, os payloads HTTP imutáveis em `raw` e finaliza o registro de controle da ingestão.

In [13]:
from pyspark.sql import functions as F

catalog = "youtube_lakehouse"
raw_schema = "raw"
silver_schema = "silver"
control_table = f"{catalog}.control.ingestion_runs"
run_error = None


def merge_silver(table_name: str, dataframe: Any, key_column: str) -> None:
    """Merge one normalized DataFrame into its silver Delta table."""
    staged_view = f"staged_{table_name}_{uuid4().hex}"
    (dataframe.withColumn("ingested_at", F.current_timestamp())
     .createOrReplaceTempView(staged_view))
    try:
        spark.sql(
            f"""
        MERGE INTO {catalog}.{silver_schema}.{table_name} AS target
        USING {staged_view} AS source
        ON target.{key_column} = source.{key_column}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
            """
        )
    finally:
        # Avoid leaking temporary views into later notebook cells.
        spark.catalog.dropTempView(staged_view)


# Recreate persistence DataFrames so display cells remain optional.
channel_df = spark.createDataFrame([channel], schema=channels_schema)
videos_df = spark.createDataFrame(videos, schema=videos_schema)
comments_df = spark.createDataFrame(comments, schema=comments_schema)
replies_df = spark.createDataFrame(replies, schema=replies_schema)

try:
    api_responses_df = spark.createDataFrame(
        api_responses,
        schema=api_responses_schema,
    )

    merge_silver("channels", channel_df, "channel_id")
    merge_silver("videos", videos_df, "video_id")
    merge_silver("comments", comments_df, "comment_id")
    merge_silver("replies", replies_df, "comment_id")
    (api_responses_df.write.mode("append").format("delta")
     .saveAsTable(f"{catalog}.{raw_schema}.api_responses"))
except Exception as exc:
    run_error = str(exc)
    raise
finally:
    status = "FAILED" if run_error else "SUCCESS"
    error_sql = "NULL" if run_error is None else (
        "'" + run_error.replace("'", "''") + "'"
    )
    spark.sql(
        f"""
        UPDATE {control_table}
        SET ended_at = current_timestamp(),
            status = '{status}',
            error_message = {error_sql}
        WHERE ingestion_id = '{ingestion_id}'
        """
    )

print(f"Ingestão {ingestion_id} finalizada com status {status}.")

Ingestão dbea2255-b48b-4541-836c-d9d55c657008 finalizada com status SUCCESS.


In [1]:
%sql
select *
from youtube_lakehouse.control.video_targets;

,video_id,is_active,priority,refresh_interval_hours,created_at,updated_at
0,dNJbFHRuHRk,True,100,24,2026-08-05 00:05:38.554680,2026-08-05 00:05:38.554680
